In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base_path = "/content/drive/MyDrive/Real Life Violence Dataset"

print("Violence videos:", len(os.listdir(os.path.join(base_path, "Violence"))))
print("NonViolence videos:", len(os.listdir(os.path.join(base_path, "NonViolence"))))


Violence videos: 1000
NonViolence videos: 1000


In [ ]:
# ===============================
# SecureVision AI – Fight Detection
# Environment Setup
# ===============================

!pip install -q torch torchvision torchaudio
!pip install -q pytorchvideo decord opencv-python tqdm
!pip install -q transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 22.1 MB/s eta 0:00:00


In [ ]:
import os
import glob
import random
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from decord import VideoReader, cpu
from tqdm import tqdm


In [ ]:
# ===============================
# Dataset Split (NO COPYING)
# ===============================

def split_dataset(base_dir, train_ratio=0.8, seed=42):
    random.seed(seed)

    samples = []

    label_map = {
        "NonViolence": 0,
        "Violence": 1
    }

    for cls, label in label_map.items():
        video_paths = []
        for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv", "*.webm"]:
          video_paths.extend(glob.glob(os.path.join(base_dir, cls, ext)))
        for vp in video_paths:
            samples.append((vp, label))

    random.shuffle(samples)

    split_idx = int(len(samples) * train_ratio)

    train_samples = samples[:split_idx]
    val_samples   = samples[split_idx:]

    print(f"Total samples: {len(samples)}")
    print(f"Train samples: {len(train_samples)}")
    print(f"Val samples:   {len(val_samples)}")

    return train_samples, val_samples


In [ ]:
# ===============================
# Dataset Root (Google Drive)
# ===============================

DATASET_ROOT = "/content/drive/MyDrive/Real Life Violence Dataset"

print("Using dataset root:", DATASET_ROOT)
print("Violence videos:", len(os.listdir(os.path.join(DATASET_ROOT, "Violence"))))
print("NonViolence videos:", len(os.listdir(os.path.join(DATASET_ROOT, "NonViolence"))))


Using dataset root: /content/drive/MyDrive/Real Life Violence Dataset
Violence videos: 1000
NonViolence videos: 1000


In [ ]:
from collections import Counter

def analyze_extensions(folder):
    files = os.listdir(folder)
    exts = [os.path.splitext(f)[1].lower() for f in files]
    return Counter(exts)

print("Violence extensions:", analyze_extensions(os.path.join(DATASET_ROOT, "Violence")))
print("NonViolence extensions:", analyze_extensions(os.path.join(DATASET_ROOT, "NonViolence")))


Violence extensions: Counter({'.mp4': 1000})
NonViolence extensions: Counter({'.mp4': 951, '.avi': 49})


In [ ]:
train_samples, val_samples = split_dataset(DATASET_ROOT)

Total samples: 2000
Train samples: 1600
Val samples:   400


In [ ]:
class FightDataset(Dataset):
    def __init__(self, samples, num_frames=16, img_size=224, mode="train"):
        """
        samples: list of (video_path, label)
        """
        self.samples = samples
        self.num_frames = num_frames
        self.mode = mode

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.samples)

    def _sample_indices(self, total_frames):
        if total_frames <= self.num_frames:
            return np.linspace(0, total_frames - 1, self.num_frames).astype(int)

        if self.mode == "train":
            start = random.randint(0, total_frames - self.num_frames)
        else:
            start = (total_frames - self.num_frames) // 2

        return np.arange(start, start + self.num_frames)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]

        try:
            vr = VideoReader(video_path, ctx=cpu(0))
            total_frames = len(vr)

            if total_frames == 0:
                raise RuntimeError("Empty video")

            indices = self._sample_indices(total_frames)
            frames = vr.get_batch(indices).asnumpy()

            frames = [self.transform(f) for f in frames]
            video = torch.stack(frames).permute(1, 0, 2, 3)  # [C, T, H, W]

            return video, torch.tensor(label)

        except Exception as e:
            # fallback: pick another random sample
            new_idx = random.randint(0, len(self.samples) - 1)
            return self.__getitem__(new_idx)


In [ ]:
# ===============================
# DataLoaders (CORRECT)
# ===============================

train_dataset = FightDataset(
    train_samples,
    num_frames=8,
    mode="train"
)

val_dataset = FightDataset(
    val_samples,
    num_frames=8,
    mode="val"
)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Train dataset size: 1600
Val dataset size: 400
Train batches: 400
Val batches: 100


In [ ]:
from transformers import TimesformerConfig, TimesformerModel

# ===============================
# TimeSformer Backbone Config
# ===============================

config = TimesformerConfig.from_pretrained(
    "facebook/timesformer-base-finetuned-k400"
)

# Explicitly match dataset
config.num_frames = 8
config.image_size = 224

backbone = TimesformerModel.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    config=config
)


In [ ]:
# ===============================
# Fight TimeSformer Model
# ===============================

class FightTimeSformer(nn.Module):
    def __init__(self, backbone, num_classes=2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(
            backbone.config.hidden_size,
            num_classes
        )

    def forward(self, x):
        outputs = self.backbone(x)
        cls_token = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_token)


model = FightTimeSformer(backbone)

# -------------------------------
# Freeze backbone parameters
# -------------------------------
for p in model.backbone.parameters():
    p.requires_grad = False

# Unfreeze last 2 encoder blocks
for block in model.backbone.encoder.layer[-2:]:
    for p in block.parameters():
        p.requires_grad = True

# Classifier always trainable
for p in model.classifier.parameters():
    p.requires_grad = True


# -------------------------------
# Device + mode handling
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# VERY IMPORTANT: keep backbone in eval mode
model.backbone.eval()

# -------------------------------
# Loss & Optimizer
# -------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,              # safer for transformers
    weight_decay=1e-4
)


In [ ]:
EPOCHS = 5  # start small

for epoch in range(EPOCHS):
    model.train()
    model.backbone.eval()   # 🔴 CRITICAL

    running_loss = 0.0

    for videos, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        videos = videos.to(device)      # [B, C, T, H, W]
        labels = labels.to(device)

        # TimeSformer expects [B, T, C, H, W]
        videos = videos.permute(0, 2, 1, 3, 4)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()

        # 🔴 Stabilize transformer training
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f}")

    # ---------- Validation ----------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            videos = videos.permute(0, 2, 1, 3, 4)

            outputs = model(videos)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = 100 * correct / total
    print(f"Epoch {epoch+1} | Val Accuracy: {val_acc:.2f}%\n")


Epoch 1/5: 100%|██████████| 400/400 [12:43<00:00,  1.91s/it]

Epoch 1 | Train Loss: 0.1636


Epoch 1 | Val Accuracy: 96.00%



Epoch 2/5: 100%|██████████| 400/400 [05:18<00:00,  1.26it/s]

Epoch 2 | Train Loss: 0.1040


...

Epoch 4 | Train Loss: 0.0590


Epoch 4 | Val Accuracy: 96.25%



Epoch 5/5: 100%|██████████| 400/400 [05:16<00:00,  1.26it/s]

Epoch 5 | Train Loss: 0.0622


Epoch 5 | Val Accuracy: 95.75%



In [ ]:
MODEL_PATH = "securevision_fight_timesformer.pth"

torch.save(model.state_dict(), MODEL_PATH)

print("Model saved at:", MODEL_PATH)


Model saved at: securevision_fight_timesformer.pth


In [ ]:
training_info = {
    "model": "TimeSformer-base-finetuned-k400",
    "num_frames": 8,
    "image_size": 224,
    "classes": ["NonViolence", "Violence"],
    "epochs": EPOCHS,
    "val_accuracy_last": val_acc
}

torch.save(training_info, "securevision_fight_timesformer_meta.pth")


In [ ]:
from google.colab import files
files.download("securevision_fight_timesformer.pth")
# files.download("securevision_fight_timesformer_meta.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/securevision_fight_timesformer.pth"

torch.save(model.state_dict(), MODEL_PATH)

print("Model saved to Google Drive at:")
print(MODEL_PATH)


Model saved to Google Drive at:
/content/drive/MyDrive/securevision_fight_timesformer.pth


In [ ]:
META_PATH = "/content/drive/MyDrive/securevision_fight_timesformer_meta.pth"

training_info = {
    "model": "TimeSformer-base-finetuned-k400",
    "num_frames": 8,
    "image_size": 224,
    "classes": ["NonViolence", "Violence"],
    "epochs": EPOCHS,
    "final_val_accuracy": val_acc
}

torch.save(training_info, META_PATH)

print("Metadata saved to Google Drive")


Metadata saved to Google Drive


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for videos, labels in val_loader:
        videos = videos.to(device)
        labels = labels.to(device)
        videos = videos.permute(0, 2, 1, 3, 4)

        outputs = model(videos)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["NonViolence", "Violence"]))
